# List of 20 countries
First, we scrape Wikipedia to get the top 20 countries by GDP, then sort them by GDP per capita.

The id of each country would also indicate its rank (GDP per capita)

In [1]:
import pandas as pd, requests as req

In [2]:
'''
    Getting the top 20 countries based on GDP (source : Wikipedia)
'''

def get_top_countries(num_countries=20):
    from io import StringIO
    import re

    url_gdp_capita = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)_per_capita"
    url_gdp = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
    url_population = "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:154.0) Gecko/20100101 Firefox/154.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
    } # To emulate firefox headers
    
    
    response = req.get(url_gdp, headers=headers)
    page_gdp = pd.read_html(StringIO(response.text))
    
    response = req.get(url_gdp_capita, headers=headers)
    page_gdp_capita = pd.read_html(StringIO(response.text))

    response = req.get(url_population, headers=headers)
    page_population = pd.read_html(StringIO(response.text))
    
    # Get the list of top GDP countries
    for i, table in enumerate(page_gdp):
        if table.shape[0]>200:
            countries = table[1:num_countries+1][["Country/Territory","IMF (2026)[1]"]] # The table includes one extra row as cumulative global GDP
            countries["Country/Territory"] = countries["Country/Territory"].map(lambda c:
                re.findall(r"^[^\[]*", c)[0])

    # Get the countries population
    for i, table in enumerate(page_population):
        if table.shape[0]>200:
            population = table[1:][["Location","Population"]] # The table includes one extra row as cumulative global population
            population = population.dropna()
            population["Location"] = population["Location"].map(lambda c:
                re.findall(r"^[^\[]*", c)[0]).str.lower()

    # Sort them based on GDP per capita
    for i, table in enumerate(page_gdp_capita):
        if table.shape[0]>200:
            table = table.dropna(subset="Country/Territory")
            table["Country/Territory"] = table["Country/Territory"].map(lambda c:
                re.findall(r"^[^\[]*", c)[0])
            countries = pd.merge(countries
                                 , table
                                 , on="Country/Territory"
                                 , suffixes=("","_capita"))

            countries["Country/Territory"] = countries["Country/Territory"].str.lower()
            countries = countries[["Country/Territory","IMF (2026)[1]","IMF (2026)[b][5]","World Bank (2025)[6]"]]
            countries = countries.rename(columns={"Country/Territory":"country"
                                      ,"IMF (2026)[1]":"imf26_gdp_milusd"
                                      ,"IMF (2026)[b][5]":"imf26_gdpcapita_usd"
                                      ,"World Bank (2025)[6]":"wb25_gdpcapita_usd"}
                             )
            countries = countries.astype({
                            "country": "string",
                            "imf26_gdp_milusd": "int",
                            "imf26_gdpcapita_usd": "int",
                        "wb25_gdpcapita_usd": "int"
                        })
            population = population.loc[population["Location"].isin(countries["country"])]
            population = population.rename(columns = {"Location": "country",
                                        "Population": "population_mil"}
                              )
            population["population_mil"] /= 10**6
            countries = pd.merge(countries
                                 , population
                                 , on =  "country")
            
            if countries["imf26_gdpcapita_usd"].isna().sum()==0:
                countries = countries.sort_values(by="imf26_gdpcapita_usd",ascending=False) ## Try to sort based on IMF26 results (if no NaN value exists)
            else:
                countries = countries.sort_values(by="wb25_gdpcapita_usd",ascending=False) ## Otherwise sort based on WB25 results

            
            countries = countries.reset_index(drop=True)
            return countries
    return None

top_economy_countries = get_top_countries()
top_economy_countries['id'] = top_economy_countries.index+1
top_economy_countries

,country,imf26_gdp_milusd,imf26_gdpcapita_usd,wb25_gdpcapita_usd,population_mil,id
0,switzerland,1135739,124478,114769,9.154896,1
1,united states,32642373,95212,90027,341.784857,2
2,netherlands,1423856,78535,73684,18.192693,3
3,australia,2110657,75172,65130,27.921150,4
4,germany,5471299,65699,60496,83.298787,5
5,united kingdom,4268446,61108,57602,69.487000,6
6,canada,2490256,60105,55698,41.798407,7
7,france,3554155,51448,48986,69.081996,8
8,italy,2701737,45886,43309,58.949500,9
9,south korea,2245847,43512,36227,51.088284,10


# WorldBank data
WorldBank's API provides access to all the datasets (sources), their sub-series (indicators), and metadata (concepts)

The endpoints to access each series in the 2nd version of their api ([https://api.worldbank.org/v2/](https://api.worldbank.org/v2/)) is given by 

    /country/{COUNTRY_ID}/indicator/{INDICATOR_ID}/source/{SOURCE_ID}/

Multiple IDs are supported using the separator (`;`). To download the available data in CSV format

    /country/{COUNTRY_ID}/indicator/{INDICATOR_ID}/source/{SOURCE_ID}?downloadformat=CSV

----

The dataset we retrieve series from is the World Development Indicators with id=2 ([https://api.worldbank.org/v2/sources](https://api.worldbank.org/v2/sources))

In [3]:
BASE_URL = "https://api.worldbank.org/v2/"
DATASET_ID = "2" ## Corresponds to World Development Indicators

## Country codes

In [4]:
response = req.get("https://api.worldbank.org/v2/country?format=json",params={"per_page":1000})
response.raise_for_status()
country_wd = pd.json_normalize(response.json()[1]) ## The 0th element is the metadata
country_wd = country_wd[["id","name"]]
country_wd = country_wd.rename(columns={"id":"codename","name":"country"})
country_wd.country = country_wd.country.str.lower()

temp_df = top_economy_countries.merge(country_wd,on="country")
temp_df

,country,imf26_gdp_milusd,imf26_gdpcapita_usd,wb25_gdpcapita_usd,population_mil,id,codename
0,switzerland,1135739,124478,114769,9.154896,1,CHE
1,united states,32642373,95212,90027,341.784857,2,USA
2,netherlands,1423856,78535,73684,18.192693,3,NLD
3,australia,2110657,75172,65130,27.921150,4,AUS
4,germany,5471299,65699,60496,83.298787,5,DEU
5,united kingdom,4268446,61108,57602,69.487000,6,GBR
6,canada,2490256,60105,55698,41.798407,7,CAN
7,france,3554155,51448,48986,69.081996,8,FRA
8,italy,2701737,45886,43309,58.949500,9,ITA
9,spain,2085276,41756,38627,49.801559,11,ESP


In [5]:
top_economy_countries.loc[~top_economy_countries.country.isin(temp_df.country),"country"]

9     south korea
13         turkey
14         russia
Name: country, dtype: object

We need to find the corresponding names

In [6]:
print(f'{"name in our data":>20} | {"available names in WorldBank":<40}')
print('-'*100)
for country in top_economy_countries.loc[~top_economy_countries.country.isin(temp_df.country),"country"]:
    print(f'{country:>20} | {" ,".join(country_wd.country[country_wd.country.str.contains(country)]):<40}')
    print('-'*100)

    name in our data | available names in WorldBank            
----------------------------------------------------------------------------------------------------
         south korea |                                         
----------------------------------------------------------------------------------------------------
              turkey |                                         
----------------------------------------------------------------------------------------------------
              russia | russian federation                      
----------------------------------------------------------------------------------------------------


Since I couldn't resolve the issue I will look for them individually.

In [7]:
print(f'{"name in our data":>20} | {"available names in WorldBank":<40}')
print('-'*100)
print(f'{"korea":>20} | {" ,".join(country_wd.country[country_wd.country.str.contains("korea")]):<40}')
print('-'*100)
print(f'{"turkey":>20} | {" ,".join(country_wd.country[country_wd.country.str.contains("tur")]):<40}')
print('-'*100)

    name in our data | available names in WorldBank            
----------------------------------------------------------------------------------------------------
               korea | korea, rep. ,korea, dem. people's rep.  
----------------------------------------------------------------------------------------------------
              turkey | turks and caicos islands ,turkmenistan ,turkiye
----------------------------------------------------------------------------------------------------


In [8]:
country_wd.loc[country_wd.country=='russian federation','country'] = 'russia'
country_wd.loc[country_wd.country=='korea, rep.','country'] = 'south korea'
country_wd.loc[country_wd.country=='turkiye','country'] = 'turkey'

In [9]:
top_economy_countries.merge(country_wd,on="country")

,country,imf26_gdp_milusd,imf26_gdpcapita_usd,wb25_gdpcapita_usd,population_mil,id,codename
0,switzerland,1135739,124478,114769,9.154896,1,CHE
1,united states,32642373,95212,90027,341.784857,2,USA
2,netherlands,1423856,78535,73684,18.192693,3,NLD
3,australia,2110657,75172,65130,27.921150,4,AUS
4,germany,5471299,65699,60496,83.298787,5,DEU
5,united kingdom,4268446,61108,57602,69.487000,6,GBR
6,canada,2490256,60105,55698,41.798407,7,CAN
7,france,3554155,51448,48986,69.081996,8,FRA
8,italy,2701737,45886,43309,58.949500,9,ITA
9,south korea,2245847,43512,36227,51.088284,10,KOR


In [10]:
top_economy_countries = top_economy_countries.merge(country_wd,on="country")

## Collecting the data

I have gathered the list of tables I need from their [Data Bank](https://databank.worldbank.org/source/world-development-indicators/) below.

In [11]:
LIST_TABLES = [
    "Current health expenditure (% of GDP)",
    "Current health expenditure per capita (current US$)",
    "Death rate, crude (per 1,000 people)",
    "Educational attainment, Doctoral or equivalent, population 25+, total (%) (cumulative)",
    "Educational attainment, at least Master's or equivalent, population 25+, total (%) (cumulative)",
    "Educational attainment, at least completed upper secondary, population 25+, total (%) (cumulative)",
    "Educational attainment, at least completed primary, population 25+ years, total (%) (cumulative)",
    "Educational attainment, at least Bachelor's or equivalent, population 25+, total (%) (cumulative)",
    "Educational attainment, at least Bachelor's or equivalent, population 25+, male (%) (cumulative)",
    "Educational attainment, at least Bachelor's or equivalent, population 25+, female (%) (cumulative)",
    "Educational attainment, at least Master's or equivalent, population 25+, male (%) (cumulative)",
    "Educational attainment, at least Master's or equivalent, population 25+, female (%) (cumulative)",
    "Educational attainment, Doctoral or equivalent, population 25+, male (%) (cumulative)",
    "Educational attainment, Doctoral or equivalent, population 25+, female (%) (cumulative)",
    "Export unit value index (2015 = 100)",
    "Export value index (2015 = 100)",
    "Export volume index (2015 = 100)",
    "Exports of goods and services (% of GDP)",
    "Exports of goods and services (annual % growth)",
    "GDP (current US$)",
    "Government expenditure on education, total (% of GDP)",
    "Labor force, total",
    "Life expectancy at birth, total (years)",
    "Primary education, pupils",
    "Primary education, pupils (% female)",
    "Primary education, pupils (% male)",
    "Population, total",
    "Secondary education, pupils",
    "Secondary education, pupils (% female)",
    "Share of youth not in education, employment or training, female (% of female youth population)",
    "Share of youth not in education, employment or training, male (% of male youth population)",
    "Share of youth not in education, employment or training, total (% of youth population)",
    "Unemployment with advanced education (% of total labor force with advanced education)",
    "Unemployment with advanced education, female (% of female labor force with advanced education)",
    "Unemployment with advanced education, male (% of male labor force with advanced education)",
    "Unemployment with basic education (% of total labor force with basic education)",
    "Unemployment with basic education, female (% of female labor force with basic education)",
    "Unemployment with basic education, male (% of male labor force with basic education)",
    "Unemployment with intermediate education (% of total labor force with intermediate education)",
    "Unemployment with intermediate education, female (% of female labor force with intermediate education)",
    "Unemployment with intermediate education, male (% of male labor force with intermediate education)"
]

**ID of each table**

In [12]:
params = {
        "format": "json",
        "per_page": 1000,
        "page": 1
    }

list_indicators = []
while True:
    response = req.get(f"https://api.worldbank.org/v2/indicators",params=params)
    response.raise_for_status()
    meta = response.json()[0]
    indicators_df = pd.json_normalize(response.json()[1])
    indicators_df = indicators_df[["id","name"]]
    list_indicators.append(indicators_df.loc[indicators_df.name.isin(LIST_TABLES)])
    if len(list_indicators[-1])==0:
        list_indicators.pop()
    
    if meta["page"] == meta["pages"]:
        break

    params["page"] += 1
    print(f"Looking at the page {meta["page"]}")

indicators_df = pd.concat(list_indicators)
indicators_df = indicators_df.sort_values(by="name")
indicators_df

Looking at the page 1
Looking at the page 2
Looking at the page 3
Looking at the page 4
Looking at the page 5
Looking at the page 6
Looking at the page 7
Looking at the page 8
Looking at the page 9
Looking at the page 10
Looking at the page 11
Looking at the page 12
Looking at the page 13
Looking at the page 14
Looking at the page 15
Looking at the page 16
Looking at the page 17
Looking at the page 18
Looking at the page 19
Looking at the page 20
Looking at the page 21
Looking at the page 22
Looking at the page 23
Looking at the page 24
Looking at the page 25
Looking at the page 26
Looking at the page 27
Looking at the page 28
Looking at the page 29


,id,name
82,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP)
83,SH.XPD.CHEX.PC.CD,Current health expenditure per capita (current...
521,SP.DYN.CDRT.IN,"Death rate, crude (per 1,000 people)"
960,SE.TER.CUAT.DO.FE.ZS,"Educational attainment, Doctoral or equivalent..."
961,SE.TER.CUAT.DO.MA.ZS,"Educational attainment, Doctoral or equivalent..."
962,SE.TER.CUAT.DO.ZS,"Educational attainment, Doctoral or equivalent..."
957,SE.TER.CUAT.BA.FE.ZS,"Educational attainment, at least Bachelor's or..."
958,SE.TER.CUAT.BA.MA.ZS,"Educational attainment, at least Bachelor's or..."
959,SE.TER.CUAT.BA.ZS,"Educational attainment, at least Bachelor's or..."
963,SE.TER.CUAT.MS.FE.ZS,"Educational attainment, at least Master's or e..."


In [13]:
indicators_df.shape, len(LIST_TABLES)

((41, 2), 41)

So now that I have gathered all the codes, I can move on to retrieve the data. I will retrieve the data based on the country's name.

**Metadata**

In [15]:
list_metadata = []
for indicator in indicators_df.id:
    response = req.get(f"https://api.worldbank.org/v2/indicator/{indicator}?format=json")
    response.raise_for_status()
    meta = response.json()[0]
    list_metadata.append(pd.json_normalize(response.json()[1]))

    print(f"Collecting metadata for {indicator}")

metadata_df = pd.concat(list_metadata)
metadata_df

,id,name,unit,sourceNote,sourceOrganization,topics,source.id,source.value
0,SH.XPD.CHEX.GD.ZS,Current health expenditure (% of GDP),,Level of current health expenditure expressed ...,"Global Health Expenditure Database, updated De...","[{'id': '8', 'value': 'Health '}]",2,World Development Indicators
0,SH.XPD.CHEX.PC.CD,Current health expenditure per capita (current...,,Current expenditures on health per capita in c...,"Global Health Expenditure Database, updated De...","[{'id': '8', 'value': 'Health '}]",2,World Development Indicators
0,SP.DYN.CDRT.IN,"Death rate, crude (per 1,000 people)",,Crude death rate indicates the number of death...,"World Population Prospects, United Nations (UN...","[{'id': '8', 'value': 'Health '}]",2,World Development Indicators
0,SE.TER.CUAT.DO.FE.ZS,"Educational attainment, Doctoral or equivalent...",,The percentage of population ages 25 and over ...,"Data API, UN Educational, Scientific and Cultu...","[{'id': '4', 'value': 'Education '}, {'id': '1...",2,World Development Indicators
0,SE.TER.CUAT.DO.MA.ZS,"Educational attainment, Doctoral or equivalent...",,The percentage of population ages 25 and over ...,"Data API, UN Educational, Scientific and Cultu...","[{'id': '4', 'value': 'Education '}, {'id': '1...",2,World Development Indicators
0,SE.TER.CUAT.DO.ZS,"Educational attainment, Doctoral or equivalent...",,The percentage of population ages 25 and over ...,"Data API, UN Educational, Scientific and Cultu...","[{'id': '4', 'value': 'Education '}, {'id': '1...",2,World Development Indicators
0,SE.TER.CUAT.BA.FE.ZS,"Educational attainment, at least Bachelor's or...",,The percentage of population ages 25 and over ...,"Data API, UN Educational, Scientific and Cultu...","[{'id': '4', 'value': 'Education '}, {'id': '1...",2,World Development Indicators
0,SE.TER.CUAT.BA.MA.ZS,"Educational attainment, at least Bachelor's or...",,The percentage of population ages 25 and over ...,"Data API, UN Educational, Scientific and Cultu...","[{'id': '4', 'value': 'Education '}, {'id': '1...",2,World Development Indicators
0,SE.TER.CUAT.BA.ZS,"Educational attainment, at least Bachelor's or...",,The percentage of population ages 25 and over ...,"Data API, UN Educational, Scientific and Cultu...","[{'id': '4', 'value': 'Education '}, {'id': '1...",2,World Development Indicators
0,SE.TER.CUAT.MS.FE.ZS,"Educational attainment, at least Master's or e...",,The percentage of population ages 25 and over ...,"Data API, UN Educational, Scientific and Cultu...","[{'id': '4', 'value': 'Education '}, {'id': '1...",2,World Development Indicators


In [16]:
metadata_df.loc[metadata_df.name.isin(metadata_df.name[metadata_df.name.duplicated()])]

,id,name,unit,sourceNote,sourceOrganization,topics,source.id,source.value
0,NE.EXP.GNFS.KD.ZG,Exports of goods and services (annual % growth),,Exports of goods includes changes in the econo...,"Country official statistics, National Statisti...","[{'id': '3', 'value': 'Economy & Growth'}, {'i...",2,World Development Indicators
0,NE.EXP.GNFS.KN.87.ZG,Exports of goods and services (annual % growth),,,,[{}],57,WDI Database Archives


Given this duplication, we have to check each table

In [17]:
for idx in ['NE.EXP.GNFS.KD.ZG','NE.EXP.GNFS.KN.87.ZG']:
    response = req.get(f"https://api.worldbank.org/v2/country/AUS/indicator/{idx}/?format=json") ## AUS=Australia is chosen randomly
    try:
        pd.json_normalize(response.json()[1])
    except Exception as e:
        print(f"Execption {e} occured")
        print(f'ID: {idx}')

Execption list index out of range occured
ID: NE.EXP.GNFS.KN.87.ZG


We have caught the table with no data

In [18]:
metadata_df = metadata_df.loc[~(metadata_df.id=='NE.EXP.GNFS.KN.87.ZG')]
indicators_df = indicators_df.loc[~(indicators_df.id=='NE.EXP.GNFS.KN.87.ZG')]

**Data**

We can focus on retrieving the data tables.

In [19]:
country_query = ';'.join(top_economy_countries.codename.values)

for indicator in indicators_df.id.values: 
    print(f"Downloding {indicator}")
    table = []
    params = {
        "format": "json",
        "per_page": 1000,
        "page": 1,
        "date": "2000:2026"
    }
    
    while True:
        response = req.get(f"https://api.worldbank.org/v2/country/{country_query}/indicator/{indicator}",params=params)
        response.raise_for_status()
        meta = response.json()[0]
        data = pd.json_normalize(response.json()[1])
        
        table.append(data)
        if len(table[-1])==0:
            table.pop()
        
        if meta["page"] == meta["pages"]:
            break
    
        params["page"] += 1
        print(f"Looking at the page {meta["page"]}")
    
    table = pd.concat(table).pivot_table(index=["countryiso3code",
                                              "indicator.id",
                                              "indicator.value",
                                              "country.id",
                                              "country.value"],
                                             columns="date",
                                            values="value",
                                            aggfunc="first").reset_index()
    table = table.rename(columns={"countryiso3code":"codename"})
    table = table.merge(top_economy_countries[["country","codename"]],on=("codename")).drop(columns=["country.value","country.id"])
    table = table[['country']+[col for col in table.columns[:-1]]]
    table.to_csv(f"Data/original/WorldBank/Data/{indicator}.csv", index=False)

Downloding SH.XPD.CHEX.GD.ZS
Downloding SH.XPD.CHEX.PC.CD
Downloding SP.DYN.CDRT.IN
Downloding SE.TER.CUAT.DO.FE.ZS
Downloding SE.TER.CUAT.DO.MA.ZS
Downloding SE.TER.CUAT.DO.ZS
Downloding SE.TER.CUAT.BA.FE.ZS
Downloding SE.TER.CUAT.BA.MA.ZS
Downloding SE.TER.CUAT.BA.ZS
Downloding SE.TER.CUAT.MS.FE.ZS
Downloding SE.TER.CUAT.MS.MA.ZS
Downloding SE.TER.CUAT.MS.ZS
Downloding SE.PRM.CUAT.ZS
Downloding SE.SEC.CUAT.UP.ZS
Downloding TX.UVI.MRCH.XD.WD
Downloding TX.VAL.MRCH.XD.WD
Downloding TX.QTY.MRCH.XD.WD
Downloding NE.EXP.GNFS.ZS
Downloding NE.EXP.GNFS.KD.ZG
Downloding NY.GDP.MKTP.CD
Downloding SE.XPD.TOTL.GD.ZS
Downloding SL.TLF.TOTL.IN
Downloding SP.DYN.LE00.IN
Downloding SP.POP.TOTL
Downloding SE.PRM.ENRL
Downloding SE.PRM.ENRL.FE.ZS
Downloding SE.SEC.ENRL
Downloding SE.SEC.ENRL.FE.ZS
Downloding SL.UEM.NEET.FE.ZS
Downloding SL.UEM.NEET.MA.ZS
Downloding SL.UEM.NEET.ZS
Downloding SL.UEM.ADVN.ZS
Downloding SL.UEM.ADVN.FE.ZS
Downloding SL.UEM.ADVN.MA.ZS
Downloding SL.UEM.BASC.ZS
Downloding S

In [20]:
top_economy_countries.to_csv("Data/cleaned/countries.csv",index=False)

In [21]:
metadata_df.to_csv("Data/cleaned/WorldBank/metadata.csv")

In [24]:
for i in metadata_df.name:
    print(i)

Current health expenditure (% of GDP)
Current health expenditure per capita (current US$)
Death rate, crude (per 1,000 people)
Educational attainment, Doctoral or equivalent, population 25+, female (%) (cumulative)
Educational attainment, Doctoral or equivalent, population 25+, male (%) (cumulative)
Educational attainment, Doctoral or equivalent, population 25+, total (%) (cumulative)
Educational attainment, at least Bachelor's or equivalent, population 25+, female (%) (cumulative)
Educational attainment, at least Bachelor's or equivalent, population 25+, male (%) (cumulative)
Educational attainment, at least Bachelor's or equivalent, population 25+, total (%) (cumulative)
Educational attainment, at least Master's or equivalent, population 25+, female (%) (cumulative)
Educational attainment, at least Master's or equivalent, population 25+, male (%) (cumulative)
Educational attainment, at least Master's or equivalent, population 25+, total (%) (cumulative)
Educational attainment, at lea

In [25]:
top_economy_countries

,country,imf26_gdp_milusd,imf26_gdpcapita_usd,wb25_gdpcapita_usd,population_mil,id,codename
0,switzerland,1135739,124478,114769,9.154896,1,CHE
1,united states,32642373,95212,90027,341.784857,2,USA
2,netherlands,1423856,78535,73684,18.192693,3,NLD
3,australia,2110657,75172,65130,27.921150,4,AUS
4,germany,5471299,65699,60496,83.298787,5,DEU
5,united kingdom,4268446,61108,57602,69.487000,6,GBR
6,canada,2490256,60105,55698,41.798407,7,CAN
7,france,3554155,51448,48986,69.081996,8,FRA
8,italy,2701737,45886,43309,58.949500,9,ITA
9,south korea,2245847,43512,36227,51.088284,10,KOR
